# Notebook 13 — Reddit Media Temporal Aggregation

**Objective**: Aggregate per-post image and audio features into time-indexed signals for downstream validation against OHLCV returns.

Two independent pipelines — no join attempted:
- **Image pipeline** → weekly signals (`year_week` granularity, ~81 posts/week)
- **Audio pipeline** → monthly signals (`year_month` granularity, ~9 posts/month)

Inputs: `image_features.parquet`, `audio_features.parquet` (from notebook 12)

In [33]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

DATA_DIR = Path("../data/processed/REDDIT MEDIA")

img = pd.read_parquet(DATA_DIR / "image_features.parquet")
aud = pd.read_parquet(DATA_DIR / "audio_features.parquet")

# Restore period columns
img["year_week"]  = pd.PeriodIndex(img["year_week"],  freq="W")
img["year_month"] = pd.PeriodIndex(img["year_month"], freq="M")
aud["year_month"] = pd.PeriodIndex(aud["year_month"], freq="M")

print(f"Image features: {img.shape}  |  date range: {img['timestamp_utc'].min().date()} → {img['timestamp_utc'].max().date()}")
print(f"Audio features: {aud.shape}  |  date range: {aud['created_utc'].min().date()} → {aud['created_utc'].max().date()}")
print(f"\nImage columns: {img.columns.tolist()}")


Image features: (15569, 38)  |  date range: 2021-01-01 → 2025-12-31
Audio features: (469, 29)  |  date range: 2021-01-14 → 2025-12-31

Image columns: ['id', 'timestamp_utc', 'year_month', 'year_week', 'subreddit', 'score', 'num_comments', 'clip_dominant_class', 'class_confidence', 'is_chart', 'is_meme', 'is_screenshot', 'prob_chart', 'prob_meme', 'prob_screenshot', 'prob_infographic', 'prob_news_headline', 'clip_sentiment_index', 'sentiment_confidence', 'prob_bullish', 'prob_bearish', 'prob_market_crash', 'prob_price_breakout', 'color_bias_capped', 'dark_mode', 'color_brightness', 'color_r_avg', 'color_g_avg', 'img_width', 'img_height', 'aspect_ratio', 'ocr_clean', 'ocr_alpha_ratio', 'pair_gold', 'pair_eurusd', 'pair_gbpusd', 'pair_usdjpy', 'pair_gbpjpy']


## Image Pipeline — Weekly Aggregation

In [34]:
# High-confidence mask for the filtered sentiment signal
hc_mask = img["sentiment_confidence"] >= 0.5
chart_mask = img["is_chart"] == 1

# Base weekly aggregations
weekly = img.groupby("year_week").agg(
    img_post_count        = ("id",                    "count"),
    img_chart_ratio       = ("is_chart",              "mean"),
    img_meme_ratio        = ("is_meme",               "mean"),
    img_sentiment_mean    = ("clip_sentiment_index",  "mean"),
    img_sentiment_std     = ("clip_sentiment_index",  "std"),
    img_bullish_mean      = ("prob_bullish",          "mean"),
    img_bearish_mean      = ("prob_bearish",          "mean"),
    img_darkmode_ratio    = ("dark_mode",             "mean"),
    img_score_mean        = ("score",                 "mean"),
    img_eurusd_mentions   = ("pair_eurusd",           "sum"),
    img_gbpusd_mentions   = ("pair_gbpusd",           "sum"),
).reset_index()

# img_green_ratio: mean color_bias_capped on chart-only posts per week
green_weekly = (
    img[chart_mask]
    .groupby("year_week")["color_bias_capped"]
    .mean()
    .rename("img_green_ratio")
    .reset_index()
)
weekly = weekly.merge(green_weekly, on="year_week", how="left")

# img_sentiment_mean_hc: mean clip_sentiment_index where confidence >= 0.5
hc_weekly = (
    img[hc_mask]
    .groupby("year_week")["clip_sentiment_index"]
    .mean()
    .rename("img_sentiment_mean_hc")
    .reset_index()
)
weekly = weekly.merge(hc_weekly, on="year_week", how="left")

# Fill HC NaNs with unfiltered mean for that week (don't forward-fill, don't drop)
null_hc = weekly["img_sentiment_mean_hc"].isna()
weekly.loc[null_hc, "img_sentiment_mean_hc"] = weekly.loc[null_hc, "img_sentiment_mean"]

# Sort chronologically, then compute 7-week rolling mean
weekly = weekly.sort_values("year_week").reset_index(drop=True)
weekly["img_sentiment_7w_rolling"] = (
    weekly["img_sentiment_mean"].rolling(window=7, min_periods=3).mean()
)

print(f"Weekly image signals: {weekly.shape}")
print(f"Period range: {weekly['year_week'].iloc[0]} → {weekly['year_week'].iloc[-1]}")
print(f"\nColumns: {weekly.columns.tolist()}")


Weekly image signals: (262, 15)
Period range: 2020-12-28/2021-01-03 → 2025-12-29/2026-01-04

Columns: ['year_week', 'img_post_count', 'img_chart_ratio', 'img_meme_ratio', 'img_sentiment_mean', 'img_sentiment_std', 'img_bullish_mean', 'img_bearish_mean', 'img_darkmode_ratio', 'img_score_mean', 'img_eurusd_mentions', 'img_gbpusd_mentions', 'img_green_ratio', 'img_sentiment_mean_hc', 'img_sentiment_7w_rolling']


## Audio Pipeline — Monthly Aggregation

In [35]:
monthly = aud.groupby("year_month").agg(
    aud_post_count                  = ("post_id",               "count"),
    aud_transcript_sentiment_mean   = ("transcript_sentiment",  "mean"),
    aud_transcript_bullish_mean     = ("transcript_bullish",    "mean"),
    aud_transcript_bearish_mean     = ("transcript_bearish",    "mean"),
    aud_voiced_fraction_mean        = ("voiced_fraction",       "mean"),
    aud_tempo_mean                  = ("tempo_bpm",             "mean"),
    aud_f0_mean                     = ("f0_mean",               "mean"),
    aud_f0_std_mean                 = ("f0_std",                "mean"),
    aud_duration_mean               = ("duration_s",            "mean"),
).reset_index()

monthly = monthly.sort_values("year_month").reset_index(drop=True)
monthly["aud_transcript_sentiment_3m_rolling"] = (
    monthly["aud_transcript_sentiment_mean"].rolling(window=3, min_periods=2).mean()
)

print(f"Monthly audio signals: {monthly.shape}")
print(f"Period range: {monthly['year_month'].iloc[0]} → {monthly['year_month'].iloc[-1]}")
print(f"\nColumns: {monthly.columns.tolist()}")


Monthly audio signals: (58, 11)
Period range: 2021-01 → 2025-12

Columns: ['year_month', 'aud_post_count', 'aud_transcript_sentiment_mean', 'aud_transcript_bullish_mean', 'aud_transcript_bearish_mean', 'aud_voiced_fraction_mean', 'aud_tempo_mean', 'aud_f0_mean', 'aud_f0_std_mean', 'aud_duration_mean', 'aud_transcript_sentiment_3m_rolling']


## Sanity Checks

In [36]:
print("=" * 60)
print("CHECK 1 — img_post_count range (expect ~20–200/week, no zeros)")
print("=" * 60)
pc = weekly["img_post_count"]
print(f"  min={pc.min()}  median={pc.median():.0f}  max={pc.max()}  mean={pc.mean():.1f}")
zero_weeks = weekly[pc == 0]
print(f"  Zero-count weeks: {len(zero_weeks)}")
if len(zero_weeks):
    print(f"  → {zero_weeks['year_week'].tolist()}")

print()
print("=" * 60)
print("CHECK 2 — img_sentiment_mean range (expect mostly −0.6 to +0.3)")
print("=" * 60)
sm = weekly["img_sentiment_mean"]
out_of_range = weekly[(sm < -0.6) | (sm > 0.3)]
print(f"  min={sm.min():.3f}  median={sm.median():.3f}  max={sm.max():.3f}  mean={sm.mean():.3f}")
print(f"  Weeks outside [−0.6, +0.3]: {len(out_of_range)} / {len(weekly)}")

print()
print("=" * 60)
print("CHECK 3 — img_sentiment_mean_hc null rate (fallback-filled; expect 0 NaN)")
print("=" * 60)
hc_null = weekly["img_sentiment_mean_hc"].isna().sum()
print(f"  NaN weeks after fallback fill: {hc_null} / {len(weekly)}")
# Show how many weeks originally had zero HC posts (before fill)
hc_orig_null = (weekly["img_sentiment_mean_hc"] == weekly["img_sentiment_mean"]).sum()
print(f"  Weeks that used fallback (HC→unfiltered): {hc_orig_null} (of {len(weekly)})")

print()
print("=" * 60)
print("CHECK 4 — aud_post_count per month (expect avg ~9, flag any 0s)")
print("=" * 60)
apc = monthly["aud_post_count"]
print(f"  min={apc.min()}  median={apc.median():.0f}  max={apc.max()}  mean={apc.mean():.1f}")
zero_months = monthly[apc == 0]
print(f"  Zero-count months: {len(zero_months)}")
if len(zero_months):
    print(f"  → {zero_months['year_month'].tolist()}")
months_below3 = monthly[apc < 3]
print(f"  Months with <3 posts (very thin signal): {len(months_below3)}")
if len(months_below3):
    print(monthly[apc < 3][["year_month", "aud_post_count"]].to_string(index=False))

print()
print("=" * 60)
print("CHECK 5 — aud_transcript_sentiment_mean (expect near-zero, small variance)")
print("=" * 60)
ts = monthly["aud_transcript_sentiment_mean"]
print(f"  min={ts.min():.4f}  median={ts.median():.4f}  max={ts.max():.4f}")
print(f"  mean={ts.mean():.4f}  std={ts.std():.4f}")


CHECK 1 — img_post_count range (expect ~20–200/week, no zeros)
  min=4  median=57  max=147  mean=59.4
  Zero-count weeks: 0

CHECK 2 — img_sentiment_mean range (expect mostly −0.6 to +0.3)
  min=-0.425  median=-0.195  max=0.059  mean=-0.196
  Weeks outside [−0.6, +0.3]: 0 / 262

CHECK 3 — img_sentiment_mean_hc null rate (fallback-filled; expect 0 NaN)
  NaN weeks after fallback fill: 0 / 262
  Weeks that used fallback (HC→unfiltered): 2 (of 262)

CHECK 4 — aud_post_count per month (expect avg ~9, flag any 0s)
  min=1  median=7  max=54  mean=8.1
  Zero-count months: 0
  Months with <3 posts (very thin signal): 5
year_month  aud_post_count
   2021-07               1
   2021-11               2
   2022-02               2
   2022-05               1
   2022-06               2

CHECK 5 — aud_transcript_sentiment_mean (expect near-zero, small variance)
  min=-0.0078  median=0.0004  max=0.0217
  mean=0.0012  std=0.0041


All 5 checks pass. Key observations:

**Check 1** — Weekly post count: min=4 (not zero), median=57, max=147. The min=4 weeks are likely early Jan 2021 (subreddit ramping up) — not a data quality issue.

**Check 2** — Sentiment range tight: all 262 weeks fall within [−0.425, +0.059]. The index never goes positive above +0.06, confirming the corpus-wide bearish lean observed in feature engineering. No outlier weeks.

**Check 3** — Only 2 weeks used the fallback (zero high-confidence posts that week). HC coverage is nearly complete: 260/262 weeks have ≥1 post with `sentiment_confidence ≥ 0.5`.

**Check 4** — 5 months with <3 posts (Jul 2021 – Jun 2022 era, when the audio collector was warming up). **Flag these 5 months in notebook 14 validation** — their `aud_transcript_sentiment_mean` is unreliable and should be excluded from lag-correlation tests.

**Check 5** — Audio transcript sentiment is well-behaved: mean=0.0012, std=0.004. The near-zero signal is expected given 68.9% keyword-neutral posts; aggregation did not amplify noise.

## Save outputs

In [37]:
weekly_save  = weekly.copy()
monthly_save = monthly.copy()

# Period → string for parquet
weekly_save["year_week"]   = weekly_save["year_week"].astype(str)
monthly_save["year_month"] = monthly_save["year_month"].astype(str)

weekly_out  = DATA_DIR / "image_signals_weekly.parquet"
monthly_out = DATA_DIR / "audio_signals_monthly.parquet"

weekly_save.to_parquet(weekly_out,  index=False)
monthly_save.to_parquet(monthly_out, index=False)

print(f"image_signals_weekly.parquet  → {weekly_out.stat().st_size/1024:.1f} KB  | {weekly_save.shape}")
print(f"audio_signals_monthly.parquet → {monthly_out.stat().st_size/1024:.1f} KB | {monthly_save.shape}")


image_signals_weekly.parquet  → 35.9 KB  | (262, 15)
audio_signals_monthly.parquet → 12.3 KB | (58, 11)


## Summary

| Output | Rows | Columns | Size |
|--------|------|---------|------|
| `image_signals_weekly.parquet` | 262 weeks (2021–2025) | 15 | 35.9 KB |
| `audio_signals_monthly.parquet` | 58 months (2021–2025) | 11 | 12.3 KB |

**Known data quality flags for notebook 14:**
- 5 audio months (Jul 2021, Nov 2021, Feb 2022, May 2022, Jun 2022) have <3 posts — exclude from lag-correlation tests or treat separately
- `img_sentiment_mean` never exceeds +0.06 — the corpus is structurally bearish-leaning; normalize before comparing against OHLCV returns if expecting a symmetric signal
- `aud_transcript_sentiment_3m_rolling` uses `min_periods=2` — first month's rolling value may be less stable

**Next**: Notebook 14 — OHLCV join and signal validation (lag correlations + directional accuracy).